# Environment & Project Sanity Check

**Updated:** 2026-03-03  
Goal: confirm Python version and that the sample CSV loads correctly.

In [36]:
import sys
sys.version

'3.12.12 (main, Mar  3 2026, 18:22:04) [Clang 17.0.0 (clang-1700.0.13.3)]'

In [3]:
import pandas as pd

df = pd.read_csv("../data/sample_pnl.csv")
df.head()


,pnl
0,0.015
1,-0.020
2,0.005
3,-0.010
4,0.012


In [4]:
print("rows:", len(df), "cols:", df.columns.tolist())

rows: 10 cols: ['pnl']


# Pandas time-series basics: rolling + dropna + alignment (risk metrics prep)

**Updated:** 2026-03-04  
Goal: build a clean time-series workflow for PnL and demonstrate why index alignment matters for rolling VaR backtests.

## 1) Load data and create a time index

We start from a simple PnL CSV and create a `DatetimeIndex` so time-series operations (rolling, alignment) behave reliably.

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

pnl = df["pnl"]  # Series with DatetimeIndex
df.head()

,pnl
date,
2026-01-01,0.015
2026-01-02,-0.020
2026-01-03,0.005
2026-01-04,-0.010
2026-01-05,0.012


## 2) Rolling windows and NaNs

`rolling(window=3)` uses the last 3 observations.  
The first `window-1` entries are `NaN` because there is not enough history to fill the window.

In [6]:
pnl_3day_mean = pnl.rolling(window=3).mean()
pnl_3day_mean

date
2026-01-01             NaN
2026-01-02             NaN
2026-01-03   -2.891206e-19
2026-01-04   -8.333333e-03
2026-01-05    2.333333e-03
2026-01-06   -9.333333e-03
2026-01-07   -3.333333e-03
2026-01-08   -9.333333e-03
2026-01-09    2.000000e-03
2026-01-10   -6.666667e-03
Name: pnl, dtype: float64

## 3) `dropna()` as a convenience (and why it can be risky)

`dropna()` removes rows with missing values.  
This is useful after rolling computations, but it can change the index/length and create alignment issues if used too early.

In [8]:
roll_clean = pnl_3day_mean.dropna()
roll_clean

date
2026-01-03   -2.891206e-19
2026-01-04   -8.333333e-03
2026-01-05    2.333333e-03
2026-01-06   -9.333333e-03
2026-01-07   -3.333333e-03
2026-01-08   -9.333333e-03
2026-01-09    2.000000e-03
2026-01-10   -6.666667e-03
Name: pnl, dtype: float64

## 4) Alignment pitfall: losing the index

Pandas aligns Series operations by index (date).  
If we convert to NumPy arrays too early (via `.to_numpy()`), we lose the index and comparisons may fail (shape mismatch) or become silently wrong.

In [9]:
alpha = 0.99
window = 3

loss = -pnl
var_roll = loss.rolling(window).quantile(alpha)
var_roll_drop = var_roll.dropna()

# BAD: dropping index + mismatched lengths (demonstration)
try:
    viol_bad = loss.to_numpy() > var_roll_drop.to_numpy()
except ValueError as e:
    print("Expected error (bad approach):", e)

Expected error (bad approach): operands could not be broadcast together with shapes (10,) (8,) 


## 5) Correct approach: keep indices and align explicitly

Two safe patterns:

1) Compare Series directly (pandas aligns by index) and then `dropna()`.  
2) Use `pd.concat(..., axis=1).dropna()` to explicitly align before comparing.

In [10]:
# GOOD 1: keep pandas alignment
viol_good1 = (loss > var_roll).dropna()
viol_good1

date
2026-01-01    False
2026-01-02    False
2026-01-03    False
2026-01-04    False
2026-01-05    False
2026-01-06     True
2026-01-07    False
2026-01-08    False
2026-01-09    False
2026-01-10     True
Name: pnl, dtype: bool

In [11]:
# GOOD 2: explicit alignment via concat
aligned = pd.concat([loss.rename("loss"), var_roll.rename("var")], axis=1).dropna()
viol_good2 = aligned["loss"] > aligned["var"]
viol_good2

date
2026-01-03    False
2026-01-04    False
2026-01-05    False
2026-01-06     True
2026-01-07    False
2026-01-08    False
2026-01-09    False
2026-01-10     True
dtype: bool

## 6) Quantile vs max (why VaR uses quantiles)

For small windows, a high quantile (e.g., 0.99) can be close to the maximum loss in the window, depending on the quantile interpolation method.
We compare `max(loss)` vs `quantile(loss, 0.99)` and show how different quantile methods change the result.

In [12]:
alpha = 0.99
window = 3

pnl_window = pnl.iloc[-window:]
loss_window = -pnl_window

loss_sorted = np.sort(loss_window.to_numpy())
max_loss = float(loss_window.max())
q_linear = float(np.quantile(loss_window.to_numpy(), alpha, method="linear"))

print("loss sorted:", loss_sorted)
print(f"max(loss) = {max_loss:.6f}")
print(f"quantile(loss,{alpha}, linear) = {q_linear:.6f}")
print(f"difference (q - max) = {q_linear - max_loss:.12f}")

loss sorted: [-0.004  0.006  0.018]
max(loss) = 0.018000
quantile(loss,0.99, linear) = 0.017760
difference (q - max) = -0.000240000000


In [45]:
aligned = pd.concat([loss.rename("loss"), var_roll.rename("var")], axis=1).dropna()
viol_good2 = aligned["loss"] > aligned["var"]


print("aligned rows:", len(aligned))
print("viol_good length:", len(viol_good2))
print(viol_good2)

aligned rows: 8
viol_good length: 8
date
2026-01-03    False
2026-01-04    False
2026-01-05    False
2026-01-06     True
2026-01-07    False
2026-01-08    False
2026-01-09    False
2026-01-10     True
dtype: bool


In [13]:
methods = ["linear", "lower", "higher", "nearest", "midpoint"]
arr = loss_window.to_numpy()

for m in methods:
    q = np.quantile(arr, alpha, method=m)
    print(f"{m:8s}: {q:.6f}")

linear  : 0.017760
lower   : 0.006000
higher  : 0.018000
nearest : 0.018000
midpoint: 0.012000


## Summary

- Time-series computations require consistent time ordering (`DatetimeIndex` + sorting).
- Rolling windows introduce `NaN` at the start; `dropna()` is helpful but should be used carefully.
- Keep pandas indices for comparisons; avoid early `.to_numpy()` when alignment matters.
- Quantile method matters for small samples and extreme confidence levels (e.g., 0.99).